In [1]:
import pandas as pd

In [2]:
data_inflacion = pd.read_csv("../data/brutos/3_BANCO MUNDIAL_INFLACION POR PAISES.csv", skiprows=4)
print(data_inflacion.shape)

(266, 71)


# 1. Me familiarizo con la estructura del dataset y observo su shape, info, head... 

In [3]:
data_inflacion.info()

<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 71 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    str    
 1   Country Code    266 non-null    str    
 2   Indicator Name  266 non-null    str    
 3   Indicator Code  266 non-null    str    
 4   1960            70 non-null     float64
 5   1961            72 non-null     float64
 6   1962            74 non-null     float64
 7   1963            75 non-null     float64
 8   1964            79 non-null     float64
 9   1965            86 non-null     float64
 10  1966            94 non-null     float64
 11  1967            101 non-null    float64
 12  1968            102 non-null    float64
 13  1969            103 non-null    float64
 14  1970            109 non-null    float64
 15  1971            113 non-null    float64
 16  1972            115 non-null    float64
 17  1973            118 non-null    float64
 18  1

In [4]:
data_inflacion.head(3) #serie histórica desde 1960 a 2025. Habrá que hacer un melt para pasar de formato años en columnas a filas como en le resto de data sets que estoy trabajando

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.028282,3.626041,4.257462,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,6.221375,4.689806,4.102851,5.191629,6.824727,10.883478,7.399186,4.770857,4.270111,NaN
2,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,4.975952,0.626149,2.302373,5.601888,5.133203,13.712102,-4.644709,-6.601186,NaN,NaN


In [5]:
print(data_inflacion.columns.tolist()[:6])
print(data_inflacion.columns.tolist()[-3:])

['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961']
['2024', '2025', 'Unnamed: 70']


# 2.  Creo la lista anios con un bucle en una sola línea: a va tomando cada número de 2015 a 2024, lo convierto en texto con str(a) y lo voy guardando en la lista (["2015", "2016", ..., "2024"]). Lo paso a texto porque así se llaman las columnas de años en el CSV del Banco Mundial, y esta lista me sirve para quedarme solo con esas columnas.


In [6]:
anios = [str(a) for a in range(2015, 2025)]
print(anios[:3], anios[-1])

['2015', '2016', '2017'] 2024


# 3 voy construyendo la estructura de la tabla, quitando las columnas que no me sirven

In [7]:
data_inflacion = data_inflacion[["Country Name", "Country Code"] + anios]
print(data_inflacion.shape)
print(data_inflacion)

(266, 12)
                    Country Name Country Code       2015       2016  \
0                          Aruba          ABW   0.474764  -0.931196   
1    Africa Eastern and Southern          AFE   5.098562   6.446877   
2                    Afghanistan          AFG  -0.661709   4.383892   
3     Africa Western and Central          AFW   2.130817   1.487416   
4                         Angola          AGO   9.355972  30.694415   
..                           ...          ...        ...        ...   
261                       Kosovo          XKX  -0.536929   0.273169   
262                  Yemen, Rep.          YEM        NaN        NaN   
263                 South Africa          ZAF   4.518691   6.602908   
264                       Zambia          ZMB  10.110593  17.869730   
265                     Zimbabwe          ZWE  -2.430968  -1.543670   

          2017       2018        2019        2020       2021        2022  \
0    -1.028282   3.626041    4.257462         NaN        NaN 

- `data_inflacion[[ ... ]]` → **doble corchete** = «quédate solo con estas columnas».
- `["Country Name","Country Code"] + anios` → suma de listas: 2 columnas fijas + las 11 de años = 13 columnas.
- Se descartan `Indicator Name`, `Indicator Code`, 1960–1999 y la columna vacía.
- Asignamos de nuevo a `data_inflacion` para «quedarnos» con el resultado.

**Debe salir:** `(266, 28)`.

# 4 Hago rename de las columnas para homogenizar con el resto de data sets que voy trabajando

In [8]:
data_inflacion = data_inflacion.rename(columns={"Country Name": "pais", "Country Code": "iso3"})
print(data_inflacion.columns.tolist()[:3])

['pais', 'iso3', '2015']


- `.rename(columns={"viejo": "nuevo"})` → cambia nombres. Es un método de **DataFrame**, con `columns=` y diccionario.
- Mismo `iso3` en los tres datasets = clave para el merge.

**Debe salir:** `['pais', 'iso3', '2015']`.

# 5 · Quito los agregados (Mundo, UE, «Ingreso alto»…) con la tabla de equivalencias, al cruzarlos
El Banco Mundial mezcla países reales con grupos. Nos quedamos solo con códigos que existen en tu tabla ISO.

In [9]:
equivalencias = pd.read_csv("../data/brutos/4_EQUIVALENCIAS_PAISES_ISO_3.csv")
print(equivalencias.shape)

(249, 4)


- Leemos la tabla de países reales (sin `skiprows`: es un CSV normal).

**Debe salir:** `(249, 4)` → columnas `codigo_onu, iso2, iso3, nombre_iso`.

In [10]:
data_inflacion = data_inflacion[data_inflacion["iso3"].isin(equivalencias["iso3"])]
print(data_inflacion.shape)

(215, 12)


- `data_inflacion["iso3"].isin(equivalencias["iso3"])` → para cada fila, `True` si su código está en la tabla de países reales, `False` si no (agregados).
- `data_inflacion[ máscara ]` → deja solo las filas `True`.
- Es el mismo patrón que `[df["col"] == valor]` del filtro de turismo, pero con «está en la lista».

**Debe salir:** `(215, 28)` → se van 51 filas (agregados y territorios que no están en tu tabla).

# 6 · Paso de formato ancho a largo con `melt`
Ahora mismo hay una columna por año. Necesitamos **una fila por país y año**.

In [11]:
data_inflacion = data_inflacion.melt(id_vars=["pais", "iso3"], var_name="anio", value_name="inflacion_(%)")
print(data_inflacion.shape)
data_inflacion.head(3)

(2150, 4)


,pais,iso3,anio,inflacion_(%)
0,Aruba,ABW,2015,0.474764
1,Afghanistan,AFG,2015,-0.661709
2,Angola,AGO,2015,9.355972


- `.melt(...)` → «derrite» las columnas de años en filas.
- `id_vars=["pais","iso3"]` → columnas que **se quedan fijas** y se repiten.
- `var_name="anio"` → nombre de la nueva columna que guarda el **nombre** de las columnas viejas (2000, 2001…).
- `value_name="inflacion (%)"` → nombre de la columna que guarda los **valores**.
- Analogía: de una tabla de doble entrada (país × año) a una lista «país, año, dato».

**Debe salir:** `(5590, 4)` = 215 países × 11 años. Primera fila: Aruba, ABW, "2000", valor.

# 7 · Convierto `anio` de string a número entero

In [12]:
data_inflacion["anio"] = data_inflacion["anio"].astype(int)
print(data_inflacion.dtypes)

pais                 str
iso3                 str
anio               int64
inflacion_(%)    float64
dtype: object


- Tras `melt`, `anio` es texto. Con texto, `2000 == "2000"` es `False` y el merge con turismo/vivienda (donde `anio` es `int`) **no casaría**.
- `.astype(int)` convierte la columna a entero.

**Debe salir:** `pais` y `iso3` como `object`/`str`, `anio` `int64`, `inflacion_pct` `float64`.

# 8 · Verifico y elimino los NaN

In [13]:
print(data_inflacion["inflacion_(%)"].isnull().sum())

339


In [14]:
data_inflacion = data_inflacion.dropna(subset=["inflacion_(%)"])
print(data_inflacion.shape)

(1811, 4)


# 9· Ordenar y reiniciar el índice

In [15]:
data_inflacion = data_inflacion.sort_values(["pais", "anio"]).reset_index(drop=True)
data_inflacion.head()

,pais,iso3,anio,inflacion_(%)
0,Afghanistan,AFG,2015,-0.661709
1,Afghanistan,AFG,2016,4.383892
2,Afghanistan,AFG,2017,4.975952
3,Afghanistan,AFG,2018,0.626149
4,Afghanistan,AFG,2019,2.302373


- `sort_values(["pais","anio"])` → ordena por país y, dentro de cada país, por año. Imprescindible para calcular variaciones por país después.
- `reset_index(drop=True)` → renumera filas 0,1,2… y descarta el índice antiguo (que quedó desordenado/con huecos).

**Debe salir:** empieza por Afghanistan 2005 (12.686…): no hay datos antes.

# 10. Redondeo el ratio de inflación a solo un decimal para dejarlo homogéneo en todos los datasets

In [16]:
data_inflacion = data_inflacion.round(1)
print(data_inflacion)

             pais iso3  anio  inflacion_(%)
0     Afghanistan  AFG  2015           -0.7
1     Afghanistan  AFG  2016            4.4
2     Afghanistan  AFG  2017            5.0
3     Afghanistan  AFG  2018            0.6
4     Afghanistan  AFG  2019            2.3
...           ...  ...   ...            ...
1806     Zimbabwe  ZWE  2018           10.6
1807     Zimbabwe  ZWE  2019          255.3
1808     Zimbabwe  ZWE  2020          557.2
1809     Zimbabwe  ZWE  2021           98.5
1810     Zimbabwe  ZWE  2022          104.7

[1811 rows x 4 columns]


# 10 · Comprobaciones finales

In [17]:
print(data_inflacion.shape)
print(data_inflacion["iso3"].nunique())
print(data_inflacion.groupby(["iso3", "anio"]).size().max())
data_inflacion[(data_inflacion["iso3"] == "ESP") & (data_inflacion["anio"] >= 2022)]

(1811, 4)
189
1


,pais,iso3,anio,inflacion_(%)
1518,Spain,ESP,2022,8.4
1519,Spain,ESP,2023,3.5
1520,Spain,ESP,2024,2.8


- `nunique()` → cuántos países distintos. **Debe salir `192`.**
- `groupby(["iso3","anio"]).size().max()` → cuenta filas por país-año; el máximo debe ser **1** (sin duplicados). Si sale >1, algo se ha repetido (¿ejecutaste una celda dos veces?).
- Última línea: filtro con dos condiciones, cada una **entre paréntesis** y unidas con `&` (no `and`).

**Debe salir para España:** 2022 → 8.39 · 2023 → 3.53 · 2024 → 2.77 · 2025 → 2.67.

## 11 · Guardo el resultado limpio

In [18]:
data_inflacion.to_csv("../data/depurados/limpio_inflacion.csv", index=False)